# M2 - Condition classification: train, evaluate, infer (Colab / Kaggle)

**Model:** Keras 3 / TensorFlow classifier `good | moderate | defective` (docs/01-prd.md §10.3).
**Plan:** docs/08-ml-plan.md §2/M2, §3.2, §5 - lightweight MobileNetV3-Small **baseline** vs
stronger EfficientNetV2-S **improved** on the *same* splits / augmentation / seed.

This notebook clones the repo and runs the existing scripts
(`ml/m2_condition_classification/scripts/`) end-to-end:
download data (Roboflow / Kaggle / upload) -> class-map -> 70/15/15 split -> train both
models -> evaluate on the frozen test split -> download the results.

**Before you start**
- Runtime -> Change runtime type -> **T4 GPU** (Colab) / Accelerator **GPU** (Kaggle). CPU works but is slow.
- Pick a license-permitted public damage/condition dataset (docs/08-ml-plan.md §3.2) and document URL + license in the report.
- In cell 1, set `REPO_URL` to your fork/repo.

Run cells in order. Skip the data cells you don't need (3A / 3B / 3C).


In [7]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
# 1. SETUP - clone repo, point M2_WORK_ROOT at persistent storage
import os, pathlib

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = os.path.exists("/content")

# TODO: your fork/repo - the one edit that matters for cloning
REPO_URL = "https://github.com/rahulpandiyan/SafeResale.git"
BRANCH   = "main"

WORK_BASE = "/kaggle/working" if IS_KAGGLE else "/content"
REPO_DIR  = os.path.join(WORK_BASE, "SafeResale")

# Export REPO_DIR as an environment variable so it's accessible by os.environ
os.environ["REPO_DIR"] = REPO_DIR

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

# Data + runs live here (mirrors ~/safresale-ml/m2 in docs/10-setup-guide.md section 7)
os.environ["M2_WORK_ROOT"] = os.path.join(WORK_BASE, "safresale-ml", "m2")
pathlib.Path(os.environ["M2_WORK_ROOT"]).mkdir(parents=True, exist_ok=True)
print("M2_WORK_ROOT =", os.environ["M2_WORK_ROOT"])

%cd {REPO_DIR}/ml/m2_condition_classification


M2_WORK_ROOT = /kaggle/working/safresale-ml/m2
/kaggle/working/SafeResale/ml/m2_condition_classification


In [9]:
# 2. DEPS + GPU CHECK (TF/Keras already ship on Colab and Kaggle)
!pip install -q pyyaml matplotlib roboflow kaggle

import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", [g.name for g in gpus] or "NONE - CPU fallback (much slower)")


TF 2.20.0 | Keras 3.13.2
GPU devices: ['/physical_device:GPU:0']


In [11]:
!pwd
!ls

/kaggle/working/SafeResale/ml/m2_condition_classification
configs  notebooks  README.md  scripts


In [12]:
%cd /kaggle/working/SafeResale
!git pull origin main

/kaggle/working/SafeResale
From https://github.com/rahulpandiyan/SafeResale
 * branch            main       -> FETCH_HEAD
Already up to date.


In [13]:
!nvidia-smi

Tue Aug 18 16:42:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Provide the dataset - run 3A AND 3B (combined), or 3C (your zip)

The model grades items as `good | moderate | defective`. No single free dataset has all
three grades, so we **combine two license-permitted sources** (configs/class_map.yaml):

| Option | Source | Feeds class | License |
|--------|--------|-------------|---------|
| 3A | Roboflow `damaged-vs-good-packages` (581 imgs, classes `damaged`/`intact`) | good + defective | CC BY 4.0 |
| 3B | Kaggle `dataclusterlabs/cracked-screen-dataset` (cracked screens) | moderate | CC0 (Public Domain) |
| 3C | Your own zip (upload / Google Drive) | any | yours |

**Run both 3A and 3B** - cell 4 merges them into one dataset. 3B is ~900 MB to download
(instant on Kaggle; slower on Colab) and needs a Kaggle account - on Colab it will ask
you to upload `kaggle.json` (https://www.kaggle.com/settings -> 'Create New API Token').

License caveat: many phone/electronics sets on Kaggle are **CC BY-NC** (non-commercial) -
avoid those for a marketplace. Note each dataset's URL + license in the report
(docs/08-ml-plan.md §3.2). Downloads land in `$M2_WORK_ROOT/data/datasets/<name>`.


In [31]:
# 3A. Roboflow Universe (classification) - damaged-vs-good-packages (CC BY 4.0)
#     Feeds: intact -> good, damaged -> defective. Free key: https://app.roboflow.com/settings/api
import os
ROBOFLOW_API_KEY = "QTXcA8ptjUufYfVJAc69"  # PASTE YOUR ROBOFLOW_API_KEY HERE
ROBOFLOW_WS   = "aadhavs-first-workspace"   # chosen dataset (damaged-vs-good-packages)
ROBOFLOW_PROJ = "damaged-vs-good-packages"  # 581 images, classes: damaged / intact
ROBOFLOW_VER  = 1

# Define the full path to the script to avoid issues with shell's current working directory
# REPO_DIR is defined in cell l4avFTJE8c-J
# Assuming `REPO_DIR` is '/kaggle/working/SafeResale' (from setup cell l4avFTJE8c-J)
# and the `download_dataset.py` script is inside 'ml/m2_condition_classification/scripts/'
script_path = os.path.join(os.environ["REPO_DIR"], "ml", "m2_condition_classification", "scripts", "download_dataset.py")

if ROBOFLOW_API_KEY and ROBOFLOW_PROJ != "mobile-phone-damage-detection":
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    DATASET_NAME = "condition_packages"
    !python {script_path} --source roboflow --workspace {ROBOFLOW_WS} --project {ROBOFLOW_PROJ} --version {ROBOFLOW_VER} --name {DATASET_NAME}
else:
    print("Skipped - paste ROBOFLOW_API_KEY above, or use 3B / 3C.")

loading Roboflow workspace...
loading Roboflow project...

Extracting Dataset Version Zip to /kaggle/working/safresale-ml/m2/data/datasets/condition_packages in folder:: 100% 592/592 [00:00<00:00, 6329.44it/s]
top-level entries: ['test', 'train', 'valid']
  train: 412 images | damaged=183, intact=229
  valid: 112 images | damaged=51, intact=61
  test: 57 images | damaged=22, intact=35
Next: remap source labels in configs/class_map.yaml, then run prepare_dataset.py.


In [32]:
# 3B. Kaggle cracked-screen set (CC0 Public Domain - commercial OK) -> moderate
#     https://www.kaggle.com/datasets/dataclusterlabs/cracked-screen-dataset
#     Needs kaggle.json (auto on Kaggle; upload on Colab). ~900 MB download.
import os
import json # Needed for creating kaggle.json

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = os.path.exists("/content")

KAGGLE_DATASET = "dataclusterlabs/cracked-screen-dataset"

# PASTE YOUR KAGGLE USERNAME AND API KEY HERE to avoid interactive file upload
KAGGLE_USERNAME = "veereshkp" # e.g., "your_kaggle_username"
KAGGLE_KEY      = "KGAT_e5ab1d1ddaf5d06bab326e68f1b7264e" # e.g., "your_kaggle_api_key_xxxxxxxxxxxxxxxxxxxxxxxx"

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

if IS_COLAB:
    # Ensure the .kaggle directory exists
    os.makedirs(os.path.dirname(kaggle_json_path), exist_ok=True)

    if not os.path.exists(kaggle_json_path):
        if KAGGLE_USERNAME and KAGGLE_KEY:
            # Create kaggle.json programmatically from provided credentials
            kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
            with open(kaggle_json_path, "w") as f:
                json.dump(kaggle_config, f)
            os.chmod(kaggle_json_path, 0o600) # Set permissions
            print("kaggle.json created from provided KAGGLE_USERNAME and KAGGLE_KEY.")
        else:
            from google.colab import files
            print("Upload kaggle.json from https://www.kaggle.com/settings -> 'Create New API Token'")
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                with open(kaggle_json_path, "wb") as f:
                    f.write(uploaded["kaggle.json"])
                os.chmod(kaggle_json_path, 0o600)
                print("kaggle.json uploaded.")
            else:
                print("No kaggle.json uploaded. Please upload the file or provide credentials.")
    else:
        print("kaggle.json already exists.")

# Define the full path to the script to avoid issues with shell's current working directory
# REPO_DIR is defined in cell l4avFTJE8c-J
# Assuming `REPO_DIR` is '/kaggle/working/SafeResale' (from setup cell l4avFTJE8c-J)
# and the `download_dataset.py` script is inside 'ml/m2_condition_classification/scripts/'
script_path = os.path.join(os.environ["REPO_DIR"], "ml", "m2_condition_classification", "scripts", "download_dataset.py")

if KAGGLE_DATASET and os.path.exists(kaggle_json_path):
    DATASET_NAME = "condition_cracked"
    !python {script_path} --source kaggle --kaggle-dataset {KAGGLE_DATASET} --name {DATASET_NAME} --as-class cracked-screen
elif KAGGLE_DATASET:
    print("Skipped Kaggle dataset download - kaggle.json not found or not created. Please provide credentials or upload the file.")
else:
    print("Skipped - set KAGGLE_DATASET, or use 3A / 3C.")

kaggle.json already exists.
$ kaggle datasets download -d dataclusterlabs/cracked-screen-dataset -p /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/.download --unzip
Dataset URL: https://www.kaggle.com/datasets/dataclusterlabs/cracked-screen-dataset
License(s): CC0-1.0
100% 858M/858M [00:42<00:00, 20.9MB/s]

collected 300 images into /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/ (class: cracked-screen)
extracted to: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked
top-level entries: ['annotations', 'cracked-screen']
Next: remap source labels in configs/class_map.yaml, then run prepare_dataset.py.


In [34]:
import os
import pathlib

# Define the full path to the script that needs modification
REPO_DIR = os.environ["REPO_DIR"]
script_path = pathlib.Path(REPO_DIR) / "ml" / "m2_condition_classification" / "scripts" / "download_dataset.py"

# Read the content of the script
with open(script_path, 'r') as f:
    lines = f.readlines()

# Modify the line that constructs the kaggle download command
# The problematic line is in the from_kaggle function
modified_lines = []
for line in lines:
    if 'cmd = [sys.executable, "-m", "kaggle", "datasets", "download",' in line:
        # Change from `sys.executable, "-m", "kaggle"` to just `"kaggle"`
        modified_lines.append(line.replace('sys.executable, "-m", "kaggle"', '"kaggle"'))
    else:
        modified_lines.append(line)

# Write the modified content back to the script
with open(script_path, 'w') as f:
    f.writelines(modified_lines)

print(f"Successfully patched {script_path} to fix Kaggle download command.")
print("Please re-run Cell 3B (Uqbbmg3o8c-O) to download the Kaggle dataset, and then Cell 4 (UxH83K5-8c-Q) to combine the datasets.")

Successfully patched /kaggle/working/SafeResale/ml/m2_condition_classification/scripts/download_dataset.py to fix Kaggle download command.
Please re-run Cell 3B (Uqbbmg3o8c-O) to download the Kaggle dataset, and then Cell 4 (UxH83K5-8c-Q) to combine the datasets.


In [35]:
# 3C. Upload your own zip (drag into the Files panel / upload dialog)
#     Expected inside the zip:  <class>/*.jpg   or   {train,valid,test}/<class>/*.jpg
import os, pathlib, zipfile
UPLOAD_ZIP = "condition_data.zip"   # TODO: name of the uploaded zip

if IS_COLAB and not os.path.exists(UPLOAD_ZIP):
    from google.colab import files
    files.upload()

if os.path.exists(UPLOAD_ZIP):
    DATASET_NAME = os.path.splitext(UPLOAD_ZIP)[0]
    dest = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "data" / "datasets" / DATASET_NAME
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as z:
        z.extractall(dest)
    print("extracted", len(list(dest.iterdir())), "top-level entries to", dest)
else:
    print("Skipped - no zip present; upload one or use 3A / 3B.")


Saving archive.zip to archive (1).zip
Saving Mobile Damage Diagnosis.v1i.yolov11.zip to Mobile Damage Diagnosis.v1i.yolov11 (1).zip
Skipped - no zip present; upload one or use 3A / 3B.


## 4. Map labels + build the split

`prepare_dataset.py` builds a **stratified 70/15/15** manifest; the test split is **frozen**
and never used in training. Imbalanced classes get inverse-frequency class weights
(no fabricated labels).

In [53]:
# 4. COMBINE SOURCES -> one dataset, map labels, then 70/15/15 split
#     (the test split is frozen and never used in training)
import os, pathlib, yaml

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
script_path_combine = m2_base_dir / "scripts" / "combine_datasets.py"
yaml_path = m2_base_dir / "configs" / "class_map.yaml"
script_path_prepare = m2_base_dir / "scripts" / "prepare_dataset.py"

# Merge the two sources into a single class-folder dataset (condition_combined):
#   packages: intact -> good, damaged -> defective
#   cracked screens (CC0): -> moderate, capped at 1000 to keep classes balanced
!python {script_path_combine} --out condition_combined     --src condition_packages/intact     --src condition_packages/damaged     --src condition_cracked/cracked-screen:1000     --overwrite

# Class-map lives in configs/class_map.yaml (committed) - show it for the report
print("class map:", yaml.safe_load(yaml_path.read_text()))

DATASET_NAME = "condition_combined"
!python {script_path_prepare} --dataset {DATASET_NAME} --out {DATASET_NAME} --class-map {yaml_path.as_posix()}

condition_packages/intact                    found 325 images
  -> /kaggle/working/safresale-ml/m2/data/datasets/condition_combined/intact (325 images)
condition_packages/damaged                   found 256 images
  -> /kaggle/working/safresale-ml/m2/data/datasets/condition_combined/damaged (256 images)
condition_cracked/cracked-screen:1000        found 299 images
  -> /kaggle/working/safresale-ml/m2/data/datasets/condition_combined/cracked-screen (299 images), cap 1000 applied

combined dataset: /kaggle/working/safresale-ml/m2/data/datasets/condition_combined
  cracked-screen   299
  damaged          256
  intact           325

Next: map labels in configs/class_map.yaml, then run:
  python scripts/prepare_dataset.py --dataset condition_combined --out condition_combined --class-map configs/class_map.yaml
class map: {'good': ['intact'], 'moderate': ['cracked-screen'], 'defective': ['damaged']}
Per-class images (source):
  defective       256
  good            325
  moderate        299



In [54]:
# 5. POINT configs/m2.yaml AT THIS MANIFEST
import os, pathlib, yaml

# Construct the full path to m2.yaml using REPO_DIR
m2_config_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification" / "configs"
cfg_path = m2_config_dir / "m2.yaml"

cfg = yaml.safe_load(cfg_path.read_text())
cfg["dataset"] = DATASET_NAME
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, default_flow_style=False))
print("configs/m2.yaml dataset =", cfg["dataset"])
print("classes =", cfg["classes"], "| models =", [m["tag"] for m in cfg["models"]])

configs/m2.yaml dataset = condition_combined
classes = ['good', 'moderate', 'defective'] | models = ['baseline-m3small', 'improved-effv2s']


### Re-checking for `InvalidArgumentError` in images

Since the training still fails with `InvalidArgumentError` despite previous cleaning and manifest regeneration, let's explicitly check each image again using TensorFlow's own decoding functions. This will help identify any *new* or *remaining* exact image files that TensorFlow is struggling to decompress.

In [55]:
import tensorflow as tf
import os
import pathlib

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")
DATASET_NAME = "condition_combined"
dataset_path = pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / DATASET_NAME

problematic_images = []

if dataset_path.exists():
    print(f"Re-checking all images in: {dataset_path} with TensorFlow's decoder...")
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg')):
                filepath = pathlib.Path(root) / file
                try:
                    # Read the image file content
                    img_bytes = tf.io.read_file(str(filepath))
                    # Attempt to decode it as a JPEG
                    _ = tf.image.decode_jpeg(img_bytes, channels=3)
                except tf.errors.InvalidArgumentError as e:
                    print(f"TensorFlow found problematic image: {filepath} - {e.message}")
                    problematic_images.append(filepath)
                except Exception as e:
                    print(f"An unexpected error occurred with {filepath}: {e}")

    if not problematic_images:
        print("No problematic JPEG images found by TensorFlow's decoder.")
        print("The `InvalidArgumentError` might be due to other factors or a transient issue within the data pipeline.")
    else:
        print(f"Found {len(problematic_images)} problematic images that TensorFlow cannot decode:")
        for img_path in problematic_images:
            print(f"  - {img_path}")
        print("You may need to manually inspect or remove these files, then re-run training.")
else:
    print(f"Dataset path not found: {dataset_path}")


Re-checking all images in: /kaggle/working/safresale-ml/m2/data/datasets/condition_combined with TensorFlow's decoder...
No problematic JPEG images found by TensorFlow's decoder.
The `InvalidArgumentError` might be due to other factors or a transient issue within the data pipeline.


### Removing problematic images from source dataset

---

\

Since the problematic images are being re-added by the `combine_datasets.py` script, we need to remove them from their original source location in the `condition_cracked` dataset.

In [57]:
import os
import pathlib

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")

# Construct full paths to the problematic images in their original source dataset
problematic_images_in_source = [
    pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / "condition_cracked" / "cracked-screen" / "cracked-screen_Datacluster Cracked Screen (217).jpg",
    pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / "condition_cracked" / "cracked-screen" / "Datacluster Cracked Screen (217).jpg"
]

print("Attempting to remove problematic image files from original source dataset...")
for img_path in problematic_images_in_source:
    if img_path.exists():
        try:
            os.remove(img_path)
            print(f"Removed from source: {img_path}")
        except OSError as e:
            print(f"Error removing {img_path} from source: {e}")
    else:
        print(f"File not found in source, skipped: {img_path}")

print("Source image removal process completed.")
print("Please now re-run Cell 4 (to re-combine datasets without these files), then re-run Cell 6A and 6B (for training).")

Attempting to remove problematic image files from original source dataset...
File not found in source, skipped: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/cracked-screen_Datacluster Cracked Screen (217).jpg
File not found in source, skipped: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/Datacluster Cracked Screen (217).jpg
Source image removal process completed.
Please now re-run Cell 4 (to re-combine datasets without these files), then re-run Cell 6A and 6B (for training).


### Correcting problematic image removal from source

In [59]:
import os
import pathlib

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")

# The identified problematic image path within its source dataset
problematic_image_source_path = pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / "condition_cracked" / "cracked-screen" / "Datacluster Cracked Screen (217).jpg"

print(f"Attempting to remove problematic image from source: {problematic_image_source_path}")

if problematic_image_source_path.exists():
    try:
        os.remove(problematic_image_source_path)
        print(f"Successfully removed: {problematic_image_source_path}")
    except OSError as e:
        print(f"Error removing {problematic_image_source_path}: {e}")
else:
    print(f"File not found at {problematic_image_source_path}. It might have already been removed or the path is incorrect.")

print("Please now re-run Cell 4 (UxH83K5-8c-Q) to re-combine datasets, then Cell 6A (1r-3IIvG8c-S) and 6B (l4uM-D1M8c-S) for training.")

Attempting to remove problematic image from source: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/Datacluster Cracked Screen (217).jpg
File not found at /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/Datacluster Cracked Screen (217).jpg. It might have already been removed or the path is incorrect.
Please now re-run Cell 4 (UxH83K5-8c-Q) to re-combine datasets, then Cell 6A (1r-3IIvG8c-S) and 6B (l4uM-D1M8c-S) for training.


## 6. Train - same protocol for both models (seed 42, identical splits + augmentation)

Two-stage schedule per docs/08-ml-plan.md §5: head on frozen base, then full fine-tune at 1/10 LR.

In [60]:
# 6A. TRAIN BASELINE - MobileNetV3-Small (lightweight)
import os, pathlib

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
train_script_path = m2_base_dir / "scripts" / "train.py"
m2_yaml_path = m2_base_dir / "configs" / "m2.yaml"

!python {train_script_path} --yaml {m2_yaml_path} --only baseline-m3small

2026-08-18 17:56:22.731667: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1787075782.733287   31561 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5

=== baseline-m3small | MobileNetV3-Small | m2-condition-mobilenetv3small-v0.1 ===
train=615 val=130 test=135 | classes=['defective', 'good', 'moderate']
devices: ['/physical_device:CPU:0', '/physical_device:GPU:0']
Epoch 1/2
2026-08-18 17:56:34.829925: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 144000000 exceeds 10% of free system memory.
I0000 00:00:1787075802.374672   31600 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s

In [61]:
# 6B. TRAIN IMPROVED - EfficientNetV2-S (stronger backbone, identical protocol)
import os, pathlib

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
train_script_path = m2_base_dir / "scripts" / "train.py"
m2_yaml_path = m2_base_dir / "configs" / "m2.yaml"

!python {train_script_path} --yaml {m2_yaml_path} --only improved-effv2s

2026-08-18 18:03:54.056781: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1787076234.058400   33812 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5

=== improved-effv2s | EfficientNetV2-S | m2-condition-efficientnetv2s-v0.1 ===
train=615 val=130 test=135 | classes=['defective', 'good', 'moderate']
devices: ['/physical_device:CPU:0', '/physical_device:GPU:0']
Epoch 1/2
2026-08-18 18:04:37.165795: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-18 18:04:37.305766: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel 

In [62]:
import os, pathlib

# 7. EVALUATE both on the FROZEN test split + per-image latency (docs/01-prd.md 10.3)
#     Writes runs/<tag>/test_metrics.json + confusion matrix PNG/CSV.
WR = os.environ["M2_WORK_ROOT"]

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
evaluate_script_path = m2_base_dir / "scripts" / "evaluate.py"

!python {evaluate_script_path} --model {WR}/runs/baseline-m3small/model.keras
!python {evaluate_script_path} --model {WR}/runs/improved-effv2s/model.keras

2026-08-18 18:18:47.196177: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1787077127.197804   37999 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787077137.420575   38043 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
model: m2-condition-mobilenetv3small-v0.1
test split: 135 images
accuracy=0.4370 | macro P=0.3345 R=0.3991 F1=0.3003
latency (per image, gpu): mean=110.91ms p50=71.11ms p95=111.92ms

per-class:
  defective    P=0.0000 R=0.0000 F1=0.0000
  good         P=0.4153 R=0.9800 F1=0.5833
  moderate     P=0.5882 R=0.2174 F1=0.3175

saved: /kaggle/working/safresale-ml/m2/runs/baseline-m3small/test_metrics.json, tes

In [63]:
# 8. SIDE-BY-SIDE METRICS (measured only - academic-integrity rule)
import json, os, pathlib
for tag in ["baseline-m3small", "improved-effv2s"]:
    p = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs" / tag / "test_metrics.json"
    print("===", tag, "===")
    if p.exists():
        m = json.loads(p.read_text())
        print(f"  accuracy={m['accuracy']:.4f}  macro P={m['precision_macro']:.4f}  R={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}")
        lat = m["latency_per_image_ms"]
        print(f"  latency mean={lat['mean_ms']:.1f}ms p50={lat['p50_ms']:.1f}ms p95={lat['p95_ms']:.1f}ms ({lat['device']})")
        for c, s in m["per_class"].items():
            print(f"    {c:10s} P={s['precision']:.4f} R={s['recall']:.4f} F1={s['f1']:.4f}")
    else:
        print("  missing - run cell 7 first")


=== baseline-m3small ===
  accuracy=0.4370  macro P=0.3345  R=0.3991  F1=0.3003
  latency mean=110.9ms p50=71.1ms p95=111.9ms (gpu)
    defective  P=0.0000 R=0.0000 F1=0.0000
    good       P=0.4153 R=0.9800 F1=0.5833
    moderate   P=0.5882 R=0.2174 F1=0.3175
=== improved-effv2s ===
  accuracy=0.8889  macro P=0.8934  R=0.8932  F1=0.8885
  latency mean=170.7ms p50=89.4ms p95=127.5ms (gpu)
    defective  P=0.7500 R=0.9231 F1=0.8276
    good       P=0.9302 R=0.8000 F1=0.8602
    moderate   P=1.0000 R=0.9565 F1=0.9778


### Inspecting `splits.json` to understand missing 'moderate' class in training

In [64]:
import os
import pathlib
import json

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")
DATASET_NAME = "condition_combined"

splits_json_path = pathlib.Path(M2_WORK_ROOT) / "data" / "condition" / DATASET_NAME / "splits.json"

print(f"Checking content of: {splits_json_path}")

if splits_json_path.exists():
    with open(splits_json_path, 'r') as f:
        splits_content = json.load(f)

    print("\nContent of splits.json:")
    print(json.dumps(splits_content, indent=2))

    # Check if 'moderate' class is present in any split
    found_moderate = False
    for split_type in ['train', 'val', 'test']:
        if split_type in splits_content:
            for item in splits_content[split_type]:
                # Corrected: item is a list, so access label by index [1]
                if len(item) > 1 and item[1] == 'moderate':
                    found_moderate = True
                    break
        if found_moderate:
            break

    if found_moderate:
        print("\n'moderate' class found in splits.json. The issue might be in how train.py interprets the manifest.")
    else:
        print("\n'moderate' class NOT found in splits.json. The issue might be in prepare_dataset.py.")

else:
    print(f"Error: splits.json not found at {splits_json_path}")

Checking content of: /kaggle/working/safresale-ml/m2/data/condition/condition_combined/splits.json

Content of splits.json:
{
  "dataset": "condition_combined",
  "source": "/kaggle/working/safresale-ml/m2/data/datasets/condition_combined",
  "classes": [
    "defective",
    "good",
    "moderate"
  ],
  "seed": 42,
  "split_ratios": [
    0.7,
    0.15,
    0.15
  ],
  "per_class": {
    "moderate": 299,
    "defective": 256,
    "good": 325
  },
  "train": [
    [
      "data/datasets/condition_combined/damaged/packagingboxeswithdamagesanddents41_webp.rf.b9deb8dd0cc706ee6414942bf5393b74.jpg",
      "defective"
    ],
    [
      "data/datasets/condition_combined/damaged/damagedfoodpackagingbox123_jpeg.rf.3bfb32d89801620bb95cfcd95942d532.jpg",
      "defective"
    ],
    [
      "data/datasets/condition_combined/damaged/packagingboxesthataredamaged173_jpeg.rf.8b394087a94828de1abfedf3420cf419.jpg",
      "defective"
    ],
    [
      "data/datasets/condition_combined/damaged/damaged

In [65]:
# 9. PACK RESULTS - download runs/ (publish weights via GitHub Releases later)
import os, pathlib, zipfile
runs = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs"
out_zip = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "m2_runs.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in runs.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(runs.parent))
print("wrote", out_zip, f"({out_zip.stat().st_size / 1e6:.1f} MB)")
if IS_COLAB:
    from google.colab import files
    files.download(str(out_zip))


wrote /kaggle/working/safresale-ml/m2/m2_runs.zip (227.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

- Publish `m2_runs.zip` weights to a **GitHub Release**; teammates run
  `scripts/infer.py --model <downloaded>/runs/<tag>/model.keras --source <img_or_folder>`
  with no dataset required (output matches the `run-vision` condition contract, docs/04-api-contract.md).
- Fill the admin **Models** page `model_metrics` from the `test_metrics.json` files above.
- Set `VISION_PROVIDER=real` + `ML_WEIGHTS_DIR` so `run-vision` uses this model (docs/03-architecture.md §5).
- Document each dataset's URL + license in the report (docs/08-ml-plan.md §3.2).
